# Preliminaires E1-E7 -- driver fin (SPEC section 5)

Ce notebook est un driver minimal : toute la logique vit dans `prelim_lib.py` (fonctions
pures), `sweep.py` (orchestration + cache/reprise) et `report.py` (generation du rapport).
Ce notebook se contente de : (1) une cellule de configuration, (2) un appel a `run_all`,
(3) un appel au generateur de rapport, (4) l'affichage de quelques figures rechargees
depuis `metrics.csv` -- le rapport (`report.md`) ne depend jamais de ces figures (SPEC
section 6), elles sont une vue complementaire, pas le livrable.

**Avant de lancer une campagne complete** : `INCLUDE_E6` ci-dessous controle le seul bloc
couteux (SPEC section 8/E6, entrainement federe reel ~30 rounds x 8 configs x 2
agregateurs) -- desactive par defaut, a activer explicitement.

In [ ]:
import os, sys, itertools
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sweep as sw
import report as rp
import prelim_lib as pl

# ---- configuration ----------------------------------------------------
INCLUDE_E6 = False   # SPEC section 8/E6, the one expensive block -- opt in explicitly
RESUME = True        # skip already-cached cells (config_id-based, see sweep.py)
CACHE_GBAR = True    # persist Gbar/grad_c to disk (float16) -- avoids ~250s/checkpoint recompute on resume

print(f"device: train={sw.TRAIN_DEVICE} eval={sw.EVAL_DEVICE}")
print(f"grid: models={sw.MODELS} seeds={sw.SEEDS} checkpoints={sw.CHECKPOINTS}")
print(f"      betas={sw.BETAS} transforms={sw.TRANSFORMS} n_p={sw.N_P} aggregators={sw.AGGREGATORS}")
print(f"include_e6={INCLUDE_E6}")

## Sweep

Remplit/complete `prelim/artifacts/metrics.csv`. Reprise automatique (par `config_id`) --
relancer cette cellule apres une interruption ne recalcule que ce qui manque.

In [ ]:
counts = sw.run_all(include_e6=INCLUDE_E6, resume=RESUME, cache_gbar=CACHE_GBAR)
counts

## Rapport

Genere `prelim/artifacts/report.md` (+ `report.json`) a partir de `metrics.csv` --
lisible sans les figures, tables uniquement (SPEC section 6).

In [ ]:
report_md = rp.build_report()
n_lines = report_md.count(chr(10)) + 1
print(f"{n_lines} lignes -> {os.path.join(rp.ARTIFACT_DIR, 'report.md')}")

## Figures

Rechargees depuis `metrics.csv` (pas depuis les objets en memoire de `run_all` -- cette
section peut etre relancee seule apres un `run_all` precedent, y compris dans une nouvelle
session). Reutilise les memes fonctions d'acces que `report.py` (`qval1`/`qvals`/
`models_present`/`checkpoints_present`/`_agg_key`) plutot que de dupliquer la logique de
filtrage. Chaque cellule est protegee par un `try/except` : des donnees partielles (ex.
`INCLUDE_E6=False`) ne font pas echouer les figures suivantes.

In [ ]:
df = pd.read_csv(sw.METRICS_PATH)
os.makedirs(sw.FIG_DIR, exist_ok=True)
MODELS_PRESENT = rp.models_present(df)
print(f"{len(df)} lignes, modeles presents: {MODELS_PRESENT}")

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(sw.FIG_DIR, name), dpi=130)
    plt.show()

### E1 -- carte de biais (implementation, transfert, signal/bruit)

In [ ]:
sub = df[df.experiment == "E1"]
if sub.empty:
    print("E1: pas de donnees")
else:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for model in MODELS_PRESENT:
            xs, ys = [], []
            for ck in rp.checkpoints_present(df, model):
                for tag in rp.E1_TAGS:
                    e = rp.qval1(df, "E1", f"relerr_shard__{tag}", model=model, checkpoint=ck)
                    c = rp.qval1(df, "E1", f"cos_shard__{tag}", model=model, checkpoint=ck)
                    if e is not None and c is not None:
                        xs.append(e); ys.append(c)
            axes[0].scatter(xs, ys, alpha=0.6, label=model)
        axes[0].axhline(0.99, color="r", ls="--", lw=1)
        axes[0].set_xlabel("err_rel (shard)"); axes[0].set_ylabel("cos (shard)")
        axes[0].set_title("E1: cos vs err_rel"); axes[0].legend(fontsize=8)

        for model in MODELS_PRESENT:
            for beta in sorted(df[(df.model == model) & (df.experiment == "E1")]["beta"].dropna().unique()):
                xs, ys = [], []
                for B in [64, 256, 1024]:
                    v = rp.qval1(df, "E1", f"sweep_err_rel__B={B}", model=model, checkpoint="end", beta=beta)
                    if v is not None:
                        xs.append(B); ys.append(v)
                if xs:
                    axes[1].plot(xs, ys, marker="o", label=f"{model}/b={beta:.2g}")
        axes[1].set_xscale("log"); axes[1].set_yscale("log")
        axes[1].set_xlabel("|B|"); axes[1].set_ylabel("erreur relative")
        axes[1].set_title("E1: erreur vs |B|"); axes[1].legend(fontsize=6)
        savefig(fig, "e1_figures.png")
    except Exception as exc:
        print(f"E1: figure ignoree ({exc})")

### E2 -- geometrie et scalaires de regime

In [ ]:
sub = df[df.experiment == "E2"]
if sub.empty:
    print("E2: pas de donnees")
else:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for model in MODELS_PRESENT:
            vals = [rp.qval1(df, "E2", f"eigval_Q_p{p}", model=model, checkpoint="end") for p in range(0, 101, 10)]
            if any(v is not None for v in vals):
                axes[0].plot([max(v, 1e-12) if v is not None else np.nan for v in vals], marker="o", ms=3, label=model)
        axes[0].set_yscale("log"); axes[0].set_title("Spectre de Q (end)"); axes[0].legend(fontsize=8)

        transforms_e2 = df[df.experiment == "E2"]["transform"].dropna().unique()
        transform = "stripe" if "stripe" in transforms_e2 else "identity"
        for model in MODELS_PRESENT:
            cks = rp.checkpoints_present(df, model)
            ys = [rp.qval1(df, "E2", "varpi", model=model, checkpoint=ck, beta=0.10, transform=transform) for ck in cks]
            xs = [ck for ck, y in zip(cks, ys) if y is not None]
            yv = [y for y in ys if y is not None]
            if yv:
                axes[1].plot(xs, yv, marker="o", label=model)
        axes[1].axhline(1.0, color="k", ls="--", lw=1)
        axes[1].set_title(f"varpi vs checkpoint (beta=0.10, T={transform})"); axes[1].legend(fontsize=8)
        savefig(fig, "e2_figures.png")
    except Exception as exc:
        print(f"E2: figure ignoree ({exc})")

### E3 -- stabilite de Gbar et cout du one-shot

In [ ]:
sub = df[df.experiment == "E3"]
if sub.empty:
    print("E3: pas de donnees")
else:
    try:
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for model in MODELS_PRESENT:
            cks = rp.checkpoints_present(df, model)
            ys = [rp.qval1(df, "E3", "l1_over_beta", model=model, checkpoint=ck) for ck in cks]
            xs = [ck for ck, y in zip(cks, ys) if y is not None]
            yv = [y for y in ys if y is not None]
            if yv:
                axes[0].plot(xs, yv, marker="o", label=model)
        axes[0].set_title("||u*||_1 / beta vs checkpoint"); axes[0].legend(fontsize=8)

        for model in MODELS_PRESENT:
            cks = rp.checkpoints_present(df, model)
            xs, ys = [], []
            for (n1, n2) in itertools.combinations(cks, 2):
                v = rp.qval1(df, "E3", "cos_u_star", model=model, checkpoint=f"{n1}_vs_{n2}")
                if v is not None:
                    xs.append(f"{n1}/{n2}"); ys.append(v)
            if ys:
                axes[1].plot(xs, ys, marker="o", label=model)
        axes[1].set_ylim(0, 1.05); axes[1].set_title("cos(u*_k, u*_k')"); axes[1].legend(fontsize=8)
        savefig(fig, "e3_figures.png")
    except Exception as exc:
        print(f"E3: figure ignoree ({exc})")

### E4 -- reponse des agregateurs robustes

In [ ]:
sub = df[df.experiment == "E4"]
if sub.empty:
    print("E4: pas de donnees")
else:
    try:
        RULES = ("mean", "cw_median", "trmean", "krum", "multikrum")
        fig, ax = plt.subplots(figsize=(6, 4))
        for model in MODELS_PRESENT:
            for rule in RULES:
                key = rp._agg_key(rule, "flat")
                vals = [rp.qval1(df, "E4", f"abar_dec{p}", model=model, transform="identity", aggregator=key)
                        for p in range(0, 101, 10)]
                if any(v is not None for v in vals):
                    ax.plot(range(0, 101, 10), vals, marker="o", ms=3, label=f"{model}/{rule}")
        ax.set_xlabel("percentile"); ax.set_ylabel("A_j"); ax.set_title("E4: Abar (flat, T=identity)")
        ax.legend(fontsize=6, ncol=2)
        savefig(fig, "e4_figures.png")
    except Exception as exc:
        print(f"E4: figure ignoree ({exc})")

### E5 -- courbe de furtivite et saturation

In [ ]:
sub = df[df.experiment == "E5"]
if sub.empty:
    print("E5: pas de donnees")
else:
    try:
        RULES = ("mean", "cw_median", "trmean", "krum", "multikrum")
        E1_BETA = sw.E1_BETA
        fig, ax = plt.subplots(figsize=(6, 4))
        for model in MODELS_PRESENT:
            taus = sorted(df[(df.model == model) & (df.experiment == "E5") &
                              (df.beta == E1_BETA) & df.tau.notna()]["tau"].unique())
            for rule in RULES:
                key = rp._agg_key(rule, "flat")
                xs, ys = [], []
                for tau in taus:
                    vh = rp.qval1(df, "E5", "v_hat", model=model, beta=E1_BETA, tau=tau)
                    s = rp.qval1(df, "E5", "selection_rate", model=model, beta=E1_BETA, tau=tau, aggregator=key)
                    if vh is not None and s is not None:
                        xs.append(vh); ys.append(s)
                if xs:
                    order = np.argsort(xs)
                    ax.plot(np.array(xs)[order], np.array(ys)[order], marker="o", ms=3, label=f"{model}/{rule}")
        ax.set_xscale("log"); ax.axvline(1.0, color="k", ls="--", lw=1)
        ax.set_xlabel("v_hat"); ax.set_ylabel("taux de selection")
        ax.set_title(f"E5: selection vs v_hat (beta={E1_BETA})"); ax.legend(fontsize=6, ncol=2)
        savefig(fig, "e5_figures.png")
    except Exception as exc:
        print(f"E5: figure ignoree ({exc})")

### E6 -- pouvoir predictif du residu (bloc couteux, INCLUDE_E6)

In [ ]:
sub = df[df.experiment == "E6"]
if sub.empty:
    print("E6: pas de donnees (bloc couteux, voir INCLUDE_E6 ci-dessus)")
else:
    try:
        PRED = rp.PREDICTOR_NAMES
        AGGS = rp.E6_AGG_NAMES
        configs = sorted(
            sub[sub.checkpoint.str.startswith("round0_p", na=False)][["checkpoint", "beta"]]
            .drop_duplicates().itertuples(index=False),
            key=lambda r: (r.checkpoint, r.beta))
        fig, axes = plt.subplots(len(AGGS), len(PRED), figsize=(4 * len(PRED), 3.5 * len(AGGS)), squeeze=False)
        for i, agg in enumerate(AGGS):
            for j, pred in enumerate(PRED):
                xs, ys = [], []
                for ck, beta in configs:
                    pair = ck.replace("round0_", "")
                    x = rp.qval1(df, "E6", pred, checkpoint=ck, beta=beta)
                    y = rp.qval1(df, "E6", "effect_rate", checkpoint=f"end_{pair}", beta=beta, aggregator=agg)
                    if x is not None and y is not None:
                        xs.append(x); ys.append(y)
                axes[i][j].scatter(xs, ys)
                axes[i][j].set_xlabel(pred)
                if j == 0:
                    axes[i][j].set_ylabel(f"{agg}\neffect_rate")
        savefig(fig, "e6_figures.png")
    except Exception as exc:
        print(f"E6: figure ignoree ({exc})")

### E7 -- etaler le budget sur n_p

In [ ]:
sub = df[df.experiment == "E7"]
if sub.empty:
    print("E7: pas de donnees")
else:
    try:
        RULES = ("mean", "cw_median", "trmean", "krum", "multikrum")
        fig, ax = plt.subplots(figsize=(6, 4))
        for model in MODELS_PRESENT:
            n_ps = sorted(df[(df.model == model) & (df.experiment == "E7")]["n_p"].dropna().unique())
            for rule in RULES:
                key = rp._agg_key(rule, "flat")
                ys = [rp.qval1(df, "E7", "selection_rate", model=model, n_p=n_p, aggregator=key) for n_p in n_ps]
                xs = [n_p for n_p, y in zip(n_ps, ys) if y is not None]
                yv = [y for y in ys if y is not None]
                if yv:
                    ax.plot(xs, yv, marker="o", label=f"{model}/{rule}")
        ax.set_xlabel("n_p"); ax.set_ylabel("taux de selection")
        ax.set_title("E7: selection vs n_p"); ax.legend(fontsize=6, ncol=2)
        savefig(fig, "e7_figures.png")
    except Exception as exc:
        print(f"E7: figure ignoree ({exc})")

## Fin

Le livrable principal est `prelim/artifacts/report.md` (+ `report.json`) genere ci-dessus --
lisible sans les figures (SPEC section 6). Les figures de cette section sont sauvegardees
sous `prelim/artifacts/figs/` en complement, pas comme dependance du rapport.